# Arctic Route RL Training — Google Colab
**사용법**: 런타임 > 런타임 유형 변경 > **T4 GPU** 선택 후 전체 실행  
별도 파일 업로드 불필요 — 모든 코드가 이 노트북에 포함되어 있음

In [ ]:
# ============================================================
# CELL 1: GPU 확인
# ============================================================
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    print(r.stdout)
else:
    raise RuntimeError('GPU 없음 — 런타임 > 런타임 유형 변경 > T4 GPU 선택 후 재실행')

In [ ]:
# ============================================================
# CELL 2: 패키지 설치
# ============================================================
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'stable-baselines3[extra]==2.3.2',
    'gymnasium==0.29.1',
])
import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('설치 완료')

In [ ]:
# ============================================================
# CELL 3: Google Drive 마운트 (결과 저장용)
# ============================================================
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/arctic_rl'
os.makedirs(f'{DRIVE_DIR}/models', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/logs', exist_ok=True)
print(f'Drive 준비: {DRIVE_DIR}')

In [ ]:
# ============================================================
# CELL 4: 모듈 코드 작성 (파일 업로드 불필요)
# ============================================================
import os
os.makedirs('/content/arctic/modules', exist_ok=True)

# ---------- config.py ----------
with open('/content/arctic/modules/config.py', 'w') as f:
    f.write('''
from __future__ import annotations
from typing import Dict, List, Tuple

ROUTE_WAYPOINTS: Dict[str, List[Tuple[float, float]]] = {
    "NSR": [
        (35.10, 129.04), (41.78, 140.81), (45.65, 141.93),
        (52.00, 155.00), (63.00, 174.00), (65.77, 169.30),
        (71.00, 180.00), (73.50, 165.00), (76.00, 140.00),
        (77.60, 104.30), (76.00, 80.00),  (72.00, 55.00),
        (70.50, 30.00),  (62.00, 5.00),   (51.90, 4.50),
    ],
    "NWP": [
        (35.10, 129.04), (45.65, 141.93), (52.00, 155.00),
        (65.77, 169.30), (71.50, -156.00),(74.30, -118.00),
        (74.00, -95.00), (72.50, -80.00), (66.50, -61.00),
        (58.00, -45.00), (51.90, 4.50),
    ],
    "TSR": [
        (35.10, 129.04), (45.65, 141.93), (65.77, 169.30),
        (75.00, 180.00), (85.00, 160.00), (88.00, 0.00),
        (80.00, -10.00), (72.00, 0.00),   (51.90, 4.50),
    ],
}

MAX_SAFE_CONCENTRATION: Dict[str, float] = {
    "PC2": 0.95, "PC3": 0.9, "PC4": 0.8, "PC5": 0.7,
    "PC6": 0.6,  "PC7": 0.5, "IA Super": 0.7, "IA": 0.6,
    "IB": 0.5,   "IC": 0.4,  "None": 0.3,
}

ICE_CLASS_FACTORS: Dict[str, float] = {
    "PC2": 0.6, "PC3": 0.7, "PC4": 0.8, "PC5": 0.9,
    "PC6": 1.0, "PC7": 1.1, "IA Super": 0.8, "IA": 0.9,
    "IB": 1.0,  "IC": 1.1,  "None": 1.3,
}
''')

# ---------- rl_ship_dynamics.py ----------
with open('/content/arctic/modules/rl_ship_dynamics.py', 'w') as f:
    f.write('''
from __future__ import annotations
import math
from dataclasses import dataclass

DEG2RAD = math.pi / 180.0
RAD2DEG = 180.0 / math.pi
EARTH_R_KM = 6_371.0
KM_PER_DEG_LAT = 111.32
NM_TO_KM = 1.852
KNOTS_TO_KMS = NM_TO_KM / 3600.0

@dataclass
class ShipState:
    lon: float = 0.0
    lat: float = 0.0
    heading: float = 0.0
    speed_knots: float = 14.0
    target_speed: float = 14.0

@dataclass
class ShipParams:
    max_speed_knots: float = 15.0
    min_speed_knots: float = 3.0
    max_turn_rate_deg_s: float = 1.5
    speed_accel_knots_s: float = 0.02
    speed_decel_knots_s: float = 0.05
    turn_rate_speed_factor: float = 0.7
    ice_drag_factor: float = 0.4

def normalize_angle(deg):
    deg = deg % 360.0
    if deg > 180.0: deg -= 360.0
    return deg

def km_per_deg_lon(lat):
    return KM_PER_DEG_LAT * math.cos(lat * DEG2RAD)

def approx_dist_km(lat1, lon1, lat2, lon2):
    d_lat = (lat2 - lat1) * KM_PER_DEG_LAT
    d_lon = (lon2 - lon1) * km_per_deg_lon((lat1 + lat2) / 2.0)
    return math.sqrt(d_lat**2 + d_lon**2)

def bearing_deg(lat1, lon1, lat2, lon2):
    d_lon = (lon2 - lon1) * km_per_deg_lon((lat1 + lat2) / 2.0)
    d_lat = (lat2 - lat1) * KM_PER_DEG_LAT
    return normalize_angle(math.atan2(d_lon, d_lat) * RAD2DEG)

def step_ship(state, params, heading_delta_deg, speed_factor, ice_concentration, dt):
    ice_drag = 1.0 - params.ice_drag_factor * min(ice_concentration, 1.0)
    target = params.max_speed_knots * speed_factor * ice_drag
    target = max(params.min_speed_knots, min(params.max_speed_knots, target))
    speed = state.speed_knots
    if speed < target:
        speed = min(target, speed + params.speed_accel_knots_s * dt)
    else:
        speed = max(target, speed - params.speed_decel_knots_s * dt)
    speed_ratio = speed / params.max_speed_knots
    effective_turn_rate = params.max_turn_rate_deg_s * (
        params.turn_rate_speed_factor + (1.0 - params.turn_rate_speed_factor) * speed_ratio)
    max_turn = effective_turn_rate * dt
    actual_turn = max(-max_turn, min(max_turn, heading_delta_deg))
    heading = normalize_angle(state.heading + actual_turn)
    speed_km_s = speed * KNOTS_TO_KMS
    dist_km = speed_km_s * dt
    heading_rad = heading * DEG2RAD
    d_north_km = dist_km * math.cos(heading_rad)
    d_east_km = dist_km * math.sin(heading_rad)
    d_lat = d_north_km / KM_PER_DEG_LAT
    cos_lat = max(0.01, math.cos(state.lat * DEG2RAD))
    d_lon = d_east_km / (KM_PER_DEG_LAT * cos_lat)
    lat = max(-89.9, min(89.9, state.lat + d_lat))
    lon = state.lon + d_lon
    if lon > 180.0: lon -= 360.0
    elif lon < -180.0: lon += 360.0
    return ShipState(lon=lon, lat=lat, heading=heading, speed_knots=speed, target_speed=target)
''')

# ---------- rl_land_mask.py ----------
with open('/content/arctic/modules/rl_land_mask.py', 'w') as f:
    f.write('''
from __future__ import annotations

_LAND_BOXES = [
    (59.0, 83.5, -73.0, -18.0),   # 그린란드
    (63.0, 67.0, -25.0, -13.0),   # 아이슬란드
    (74.0, 81.0, 10.0, 33.0),     # 스발바르
    (70.0, 77.5, 51.0, 68.5),     # 노바야 제믈랴
    (78.0, 81.5, 91.0, 107.0),    # 세베르나야 제믈랴
    (79.5, 82.0, 44.0, 65.0),     # 프란츠 요제프
    (73.0, 76.5, 136.0, 159.0),   # 뉴시베리아
    (70.5, 71.5, -179.0, -178.0), # 브란겔 서
    (70.5, 71.5, 178.5, 180.0),   # 브란겔 동
    (72.0, 84.0, -90.0, -61.0),   # 배핀 섬 북부
    (68.0, 74.0, -115.0, -95.0),  # 빅토리아 섬
    (65.0, 71.5, -165.0, -145.0), # 알래스카 북부
    (58.0, 71.5, 5.0, 28.0),      # 스칸디나비아 북부
    (73.0, 77.5, 92.0, 106.0),    # 타이미르 반도
    (69.5, 73.0, 67.0, 73.0),     # 야말 반도
    (64.5, 67.5, 172.0, 180.0),   # 축치 반도
    (51.5, 59.0, 160.5, 163.0),   # 캄차카
    (43.5, 45.5, 141.5, 145.0),   # 홋카이도
    (46.0, 54.0, 141.8, 143.2),   # 사할린
]

_WAYPOINT_WHITELIST = [
    (45.65, 141.93, 0.5),   # 소야 해협
    (41.78, 140.81, 0.5),   # 쓰가루 해협
    (74.30, -118.00, 1.5),  # 맥클루어 해협
    (72.50, -80.00, 1.5),   # 랭커스터 해협
    (66.50, -61.00, 1.5),   # 배핀만 입구
    (71.00, 180.00, 1.0),   # 척치해 동
    (71.00, -180.00, 1.0),  # 척치해 서
    (71.50, -156.00, 1.5),  # 척치-보퍼트해
    (65.77, 169.30, 0.5),   # 베링 해협
    (77.60, 104.30, 1.5),   # 빌키츠키 해협
    (72.00, 55.00, 1.5),    # 바렌츠해 입구
    (76.00, 80.00, 1.5),    # 카라해
    (76.00, 140.00, 1.5),   # 랍테프해
    (73.50, 165.00, 1.5),   # 뉴시베리아 인근
    (58.00, -45.00, 1.0),   # 래브라도해
    (62.00, 5.00, 1.5),     # 노르웨이해
    (74.00, -95.00, 1.5),   # 바이카운트멜빌
]

class LandMask:
    def is_land(self, lat, lon):
        lon_n = ((lon + 180.0) % 360.0) - 180.0
        for wp_lat, wp_lon, radius in _WAYPOINT_WHITELIST:
            if abs(lat - wp_lat) <= radius and abs(lon_n - wp_lon) <= radius:
                return False
        for lat_min, lat_max, lon_min, lon_max in _LAND_BOXES:
            if lat_min <= lat <= lat_max and lon_min <= lon_n <= lon_max:
                return True
        return False
''')

# ---------- rl_reward.py ----------
with open('/content/arctic/modules/rl_reward.py', 'w') as f:
    f.write('''
from __future__ import annotations
import math
from dataclasses import dataclass
from .config import MAX_SAFE_CONCENTRATION, ICE_CLASS_FACTORS

@dataclass
class RewardWeights:
    collision: float = -50.0
    proximity: float = -0.5
    danger_zone: float = -1.0
    route_deviation: float = -0.5
    progress: float = 10.0
    smoothness: float = -0.01
    fuel: float = -0.005
    ice_concentration: float = -0.1
    episode_success: float = 200.0

@dataclass
class RewardContext:
    ship_lat: float
    ship_lon: float
    ship_speed_knots: float
    heading_change_deg: float
    speed_factor: float
    iceberg_distances_km: list
    iceberg_sizes_m: list
    cross_track_error_km: float
    along_track_progress: float
    max_allowed_deviation_km: float = 30.0
    ice_concentration: float = 0.0
    max_safe_concentration: float = 0.7
    visibility_km: float = 10.0
    wave_height_m: float = 1.0
    collision: bool = False
    episode_done_success: bool = False

def compute_dynamic_safety_radius(base_radius_km, speed_knots, visibility_km, ice_class_factor=1.0):
    speed_scale = max(0.5, speed_knots / 12.0)
    visibility_scale = 1.0 / max(visibility_km, 1.0)
    return base_radius_km * speed_scale * (1.0 + visibility_scale) * ice_class_factor

def compute_reward(ctx, weights=None):
    if weights is None:
        weights = RewardWeights()
    components = {}
    components["collision"] = weights.collision if ctx.collision else 0.0
    safety_radius = compute_dynamic_safety_radius(10.0, ctx.ship_speed_knots, ctx.visibility_km)
    proximity_penalty = 0.0
    danger_zone_penalty = 0.0
    for i, dist_km in enumerate(ctx.iceberg_distances_km):
        size_m = ctx.iceberg_sizes_m[i] if i < len(ctx.iceberg_sizes_m) else 5000.0
        size_factor = min(2.0, size_m / 5000.0)
        collision_r = max(0.5, size_m / 1000.0 / 2.0)
        if dist_km < collision_r * 2.0:
            danger_zone_penalty = danger_zone_penalty + float(size_factor * (1.0 - dist_km / (collision_r * 2.0)))
        elif dist_km < safety_radius * 3:
            proximity_penalty = proximity_penalty + float(math.exp(-(dist_km / safety_radius)**2) * size_factor)
    components["proximity"] = weights.proximity * proximity_penalty
    components["danger_zone"] = weights.danger_zone * danger_zone_penalty
    deviation_ratio = min(1.0, abs(ctx.cross_track_error_km) / ctx.max_allowed_deviation_km)
    components["route_deviation"] = weights.route_deviation * deviation_ratio
    if ctx.along_track_progress > 1e-5:
        components["progress"] = weights.progress
    elif ctx.along_track_progress < -1e-5:
        components["progress"] = weights.progress * 0.5
    else:
        components["progress"] = 0.0
    turn_ratio = abs(ctx.heading_change_deg) / 15.0
    components["smoothness"] = weights.smoothness * min(1.0, turn_ratio)
    components["fuel"] = weights.fuel * (1.0 - ctx.speed_factor)
    if ctx.max_safe_concentration > 0:
        ice_ratio = ctx.ice_concentration / ctx.max_safe_concentration
        components["ice_concentration"] = weights.ice_concentration * min(1.0, ice_ratio)
    else:
        components["ice_concentration"] = 0.0
    components["episode_success"] = weights.episode_success if ctx.episode_done_success else 0.0
    total = sum(components.values())
    return total, components
''')

print('모듈 파일 생성 완료 (config, ship_dynamics, land_mask, reward)')

In [ ]:
# ============================================================
# CELL 5: 환경 모듈 작성
# ============================================================
with open('/content/arctic/modules/rl_environment.py', 'w') as f:
    f.write('''
from __future__ import annotations
import math, random
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from .rl_ship_dynamics import (
    ShipState, ShipParams, step_ship,
    approx_dist_km, bearing_deg, normalize_angle,
    KM_PER_DEG_LAT, km_per_deg_lon,
)
from .rl_reward import RewardContext, RewardWeights, compute_reward, compute_dynamic_safety_radius
from .rl_land_mask import LandMask
from .config import ROUTE_WAYPOINTS, MAX_SAFE_CONCENTRATION

class Iceberg:
    __slots__ = ("lat", "lon", "length_m", "width_m")
    def __init__(self, lat, lon, length_m=5000.0, width_m=3000.0):
        self.lat, self.lon, self.length_m, self.width_m = lat, lon, length_m, width_m

def _random_icebergs_along_segment(lat1, lon1, lat2, lon2, count, spread_km=35.0):
    bergs = []
    for _ in range(count):
        t = random.random()
        clat = lat1 + t * (lat2 - lat1)
        clon = lon1 + t * (lon2 - lon1)
        olat = random.gauss(0, spread_km / KM_PER_DEG_LAT / 3)
        olon = random.gauss(0, spread_km / max(1, km_per_deg_lon(clat)) / 3)
        st = random.choices(["small","medium","large","tabular"], weights=[0.4,0.3,0.2,0.1])[0]
        sizes = {"small":(random.uniform(25,80),random.uniform(15,50)),
                 "medium":(random.uniform(80,200),random.uniform(50,120)),
                 "large":(random.uniform(200,500),random.uniform(100,300)),
                 "tabular":(random.uniform(500,2000),random.uniform(300,1000))}
        lm, wm = sizes[st]
        bergs.append(Iceberg(lat=clat+olat, lon=clon+olon, length_m=lm, width_m=wm))
    return bergs

def _cross_track_error(ship_lat, ship_lon, wp1, wp2):
    lat1, lon1 = wp1; lat2, lon2 = wp2
    dAB_n = (lat2-lat1)*KM_PER_DEG_LAT
    dAB_e = (lon2-lon1)*km_per_deg_lon((lat1+lat2)/2)
    dAP_n = (ship_lat-lat1)*KM_PER_DEG_LAT
    dAP_e = (ship_lon-lon1)*km_per_deg_lon((lat1+ship_lat)/2)
    AB_len = math.sqrt(dAB_n**2+dAB_e**2)
    if AB_len < 1e-6: return approx_dist_km(ship_lat,ship_lon,lat1,lon1)
    return (dAP_e*dAB_n - dAP_n*dAB_e) / AB_len

def _along_track_fraction(ship_lat, ship_lon, wp1, wp2):
    lat1, lon1 = wp1; lat2, lon2 = wp2
    dAB_n = (lat2-lat1)*KM_PER_DEG_LAT
    dAB_e = (lon2-lon1)*km_per_deg_lon((lat1+lat2)/2)
    dAP_n = (ship_lat-lat1)*KM_PER_DEG_LAT
    dAP_e = (ship_lon-lon1)*km_per_deg_lon((lat1+ship_lat)/2)
    AB2 = dAB_n**2+dAB_e**2
    if AB2 < 1e-6: return 0.0
    return max(0.0, min(1.0, (dAP_n*dAB_n+dAP_e*dAB_e)/AB2))

class IcebergAvoidanceEnv(gym.Env):
    metadata = {"render_modes": ["human"]}
    MAX_STEPS = 3000
    DT = 2.0
    MAX_DEVIATION_KM = 50.0
    COLLISION_RADIUS_KM = 0.5
    MAX_NEARBY_ICEBERGS = 3
    SEGMENT_MAX_DIST_KM = 40.0
    SUCCESS_PROGRESS = 0.90

    def __init__(self, render_mode=None, difficulty="medium", reward_weights=None,
                 fixed_route=None, fixed_ice_class=None, ship_params=None):
        super().__init__()
        self.render_mode = render_mode
        self.difficulty = difficulty
        self._fixed_route = fixed_route
        self._fixed_ice_class = fixed_ice_class
        self._custom_ship_params = ship_params
        self.action_space = spaces.Box(
            low=np.array([-15.0, 0.5], dtype=np.float32),
            high=np.array([15.0, 1.0], dtype=np.float32))
        self.observation_space = spaces.Box(
            low=-np.ones(22, dtype=np.float32)*2.0,
            high=np.ones(22, dtype=np.float32)*2.0)
        self.ship = None
        self.ship_params = ship_params if ship_params else ShipParams()
        self.reward_weights = reward_weights if reward_weights else RewardWeights()
        self.icebergs = []
        self.route_wps = []
        self.segment_start_idx = 0
        self.segment_end_idx = 0
        self.ice_class = "PC5"
        self.max_safe_conc = 0.7
        self.ice_concentration = 0.0
        self.visibility_km = 10.0
        self.wave_height_m = 1.0
        self.step_count = 0
        self.prev_progress = 0.0
        self.land_mask = LandMask()

    def _get_difficulty_params(self):
        if self.difficulty == "easy":
            return {"berg_count":(0,0),"ice_conc":(0.0,0.05),"visibility":(18,20),"wave":(0.0,0.3)}
        elif self.difficulty == "medium":
            return {"berg_count":(3,8),"ice_conc":(0.1,0.3),"visibility":(8,15),"wave":(0.5,1.5)}
        else:
            return {"berg_count":(8,20),"ice_conc":(0.2,0.6),"visibility":(3,8),"wave":(1.0,3.0)}

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        route_key = self._fixed_route if self._fixed_route else random.choice(list(ROUTE_WAYPOINTS.keys()))
        self.route_wps = ROUTE_WAYPOINTS[route_key]
        n = len(self.route_wps)
        for _attempt in range(20):
            seg_len = random.randint(1, min(3, n-1))
            start_idx = random.randint(0, n-seg_len-1)
            end_idx = start_idx + seg_len
            seg_dist = sum(approx_dist_km(self.route_wps[i][0],self.route_wps[i][1],
                                          self.route_wps[i+1][0],self.route_wps[i+1][1])
                          for i in range(start_idx, end_idx))
            if seg_dist <= self.SEGMENT_MAX_DIST_KM: break
        self.segment_start_idx = start_idx
        self.segment_end_idx = end_idx
        self.ice_class = self._fixed_ice_class if self._fixed_ice_class else random.choice(["PC3","PC5","PC7","IA Super","IA"])
        self.max_safe_conc = MAX_SAFE_CONCENTRATION.get(self.ice_class, 0.7)
        dp = self._get_difficulty_params()
        berg_count = random.randint(*dp["berg_count"])
        self.ice_concentration = random.uniform(*dp["ice_conc"])
        self.visibility_km = random.uniform(*dp["visibility"])
        self.wave_height_m = random.uniform(*dp["wave"])
        self.icebergs = []
        if berg_count > 0:
            for i in range(self.segment_start_idx, self.segment_end_idx):
                wp1 = self.route_wps[i]; wp2 = self.route_wps[i+1]
                count_in_seg = max(1, berg_count // (end_idx - start_idx))
                self.icebergs.extend(_random_icebergs_along_segment(
                    wp1[0],wp1[1],wp2[0],wp2[1],count_in_seg,spread_km=35.0))
        start_wp = self.route_wps[self.segment_start_idx]
        next_wp = self.route_wps[self.segment_start_idx+1]
        initial_heading = bearing_deg(start_wp[0],start_wp[1],next_wp[0],next_wp[1])
        self.ship = ShipState(lon=start_wp[1],lat=start_wp[0],heading=initial_heading,speed_knots=14.0,target_speed=14.0)
        self.step_count = 0
        self.prev_progress = 0.0
        return self._get_obs(), {}

    def _get_progress(self):
        seg_dists = []
        total_dist = 0.0
        for i in range(self.segment_start_idx, self.segment_end_idx):
            d = float(approx_dist_km(self.route_wps[i][0],self.route_wps[i][1],
                                     self.route_wps[i+1][0],self.route_wps[i+1][1]))
            seg_dists.append(d); total_dist += d
        if total_dist < 1e-3: return 0.0
        best_cum = 0.0; cum_before = 0.0
        for k, d in enumerate(seg_dists):
            i = self.segment_start_idx + k
            frac = float(_along_track_fraction(self.ship.lat,self.ship.lon,self.route_wps[i],self.route_wps[i+1]))
            candidate = cum_before + frac * d
            if candidate > best_cum: best_cum = candidate
            if frac < 1.0: break
            cum_before += d
        return min(1.0, float(best_cum)/float(total_dist))

    def _get_cross_track(self):
        best_xt = float("inf")
        for i in range(self.segment_start_idx, self.segment_end_idx):
            xt = _cross_track_error(self.ship.lat,self.ship.lon,self.route_wps[i],self.route_wps[i+1])
            if abs(xt) < abs(best_xt): best_xt = xt
        return best_xt

    def _nearest_icebergs(self, n=3):
        dists = []
        for berg in self.icebergs:
            d = approx_dist_km(self.ship.lat,self.ship.lon,berg.lat,berg.lon)
            b = bearing_deg(self.ship.lat,self.ship.lon,berg.lat,berg.lon)
            rb = normalize_angle(b - self.ship.heading)
            dists.append((rb,d,berg.length_m))
        dists.sort(key=lambda x: x[1])
        result = dists[:n]
        while len(result) < n: result.append((0.0,999.0,0.0))
        return result

    def _get_obs(self):
        obs = np.zeros(22, dtype=np.float32)
        obs[0] = self.ship.lon/180.0; obs[1] = self.ship.lat/90.0
        h_rad = self.ship.heading*math.pi/180.0
        obs[2] = math.sin(h_rad); obs[3] = math.cos(h_rad)
        obs[4] = self.ship.speed_knots/self.ship_params.max_speed_knots
        target_idx = self.segment_start_idx+1
        for i in range(self.segment_start_idx+1, self.segment_end_idx+1):
            wp = self.route_wps[i]
            frac = _along_track_fraction(self.ship.lat,self.ship.lon,self.route_wps[i-1],wp)
            if frac < 0.95: target_idx = i; break
        target_wp = self.route_wps[min(target_idx, len(self.route_wps)-1)]
        obs[5] = (target_wp[1]-self.ship.lon)*km_per_deg_lon(self.ship.lat)/100.0
        obs[6] = (target_wp[0]-self.ship.lat)*KM_PER_DEG_LAT/100.0
        dist_to_wp = approx_dist_km(self.ship.lat,self.ship.lon,target_wp[0],target_wp[1])
        bearing_to_wp = bearing_deg(self.ship.lat,self.ship.lon,target_wp[0],target_wp[1])
        obs[7] = normalize_angle(bearing_to_wp-self.ship.heading)/180.0
        obs[8] = min(1.0,dist_to_wp/200.0)
        nearest = self._nearest_icebergs(self.MAX_NEARBY_ICEBERGS)
        for i,(rb,d,sz) in enumerate(nearest):
            obs[9+i*2] = rb/180.0; obs[10+i*2] = min(1.0,d/50.0)
        obs[15] = min(1.0,self.ice_concentration)
        obs[16] = min(1.0,self.max_safe_conc)
        obs[17] = min(1.0,self.visibility_km/20.0)
        obs[18] = min(1.0,self.wave_height_m/8.0)
        obs[19] = self.max_safe_conc
        obs[20] = self._get_progress()
        xt = self._get_cross_track()
        obs[21] = max(-1.0,min(1.0,xt/self.MAX_DEVIATION_KM))
        return obs

    def step(self, action):
        heading_delta = float(np.clip(action[0],-15.0,15.0))
        speed_factor = float(np.clip(action[1],0.5,1.0))
        self.ship = step_ship(self.ship,self.ship_params,heading_delta,speed_factor,self.ice_concentration,self.DT)
        self.step_count += 1
        collision = False
        iceberg_dists = []; iceberg_sizes = []
        for berg in self.icebergs:
            d = approx_dist_km(self.ship.lat,self.ship.lon,berg.lat,berg.lon)
            iceberg_dists.append(d); iceberg_sizes.append(berg.length_m)
            collision_r = max(self.COLLISION_RADIUS_KM, berg.length_m/1000.0/2.0)
            if d < collision_r: collision = True
        if not collision and self.land_mask.is_land(self.ship.lat,self.ship.lon):
            collision = True
        current_progress = self._get_progress()
        delta_progress = current_progress - self.prev_progress
        self.prev_progress = current_progress
        xt_km = abs(self._get_cross_track())
        terminated = False; truncated = False; success = False
        if collision: terminated = True
        elif current_progress >= self.SUCCESS_PROGRESS: terminated = True; success = True
        elif xt_km > self.MAX_DEVIATION_KM: terminated = True
        elif self.step_count >= self.MAX_STEPS: truncated = True
        ctx = RewardContext(
            ship_lat=self.ship.lat,ship_lon=self.ship.lon,
            ship_speed_knots=self.ship.speed_knots,
            heading_change_deg=heading_delta,speed_factor=speed_factor,
            iceberg_distances_km=iceberg_dists,iceberg_sizes_m=iceberg_sizes,
            cross_track_error_km=xt_km,along_track_progress=delta_progress,
            max_allowed_deviation_km=self.MAX_DEVIATION_KM,
            ice_concentration=self.ice_concentration,max_safe_concentration=self.max_safe_conc,
            visibility_km=self.visibility_km,wave_height_m=self.wave_height_m,
            collision=collision,episode_done_success=success)
        reward, reward_components = compute_reward(ctx, self.reward_weights)
        obs = self._get_obs()
        info = {"collision":collision,"success":success,"progress":current_progress,
                "cross_track_km":xt_km,"reward_components":reward_components,"step":self.step_count}
        return obs, reward, terminated, truncated, info

    def get_ship_state(self):
        return {"lon":self.ship.lon,"lat":self.ship.lat,"heading":self.ship.heading,"speed_knots":self.ship.speed_knots}
''')

# ---------- __init__.py ----------
with open('/content/arctic/modules/__init__.py', 'w') as f:
    f.write('')

print('환경 모듈 생성 완료')

In [ ]:
# ============================================================
# CELL 6: 에이전트 / 트레이너 모듈 작성
# ============================================================
import sys
sys.path.insert(0, '/content/arctic')

# ---------- rl_agent.py ----------
with open('/content/arctic/modules/rl_agent.py', 'w') as f:
    f.write('''
from __future__ import annotations
import json, logging
from collections import deque
from itertools import islice
from pathlib import Path
from typing import Optional
import numpy as np
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import BaseCallback
from gymnasium.wrappers import RecordEpisodeStatistics
from .rl_environment import IcebergAvoidanceEnv
from .rl_reward import RewardWeights

class _StopTraining(Exception): pass

logger = logging.getLogger(__name__)

DEFAULT_HYPERPARAMS = {
    "learning_rate": 3e-4,
    "buffer_size": 300_000,
    "batch_size": 256,
    "gamma": 0.95,
    "tau": 0.005,
    "ent_coef": "auto",
    "train_freq": 1,
    "gradient_steps": 1,
    "learning_starts": 1_000,
    "policy_kwargs": {"net_arch": [256, 256]},
}

class TrainingMetricsCallback(BaseCallback):
    def __init__(self, log_interval=5000, verbose=0):
        super().__init__(verbose)
        self.log_interval = log_interval
        self.episode_rewards = deque(maxlen=2000)
        self.collision_count = 0
        self.success_count = 0
        self.total_episodes = 0
        self.metrics_history = deque(maxlen=500)

    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.total_episodes += 1
                self.episode_rewards.append(float(info["episode"]["r"]))
            if info.get("collision"): self.collision_count += 1
            if info.get("success"): self.success_count += 1
        if self.num_timesteps % self.log_interval == 0 and self.total_episodes > 0:
            recent = list(islice(reversed(self.episode_rewards), 100))
            metrics = {
                "timestep": self.num_timesteps,
                "episodes": self.total_episodes,
                "mean_reward_100": float(np.mean(recent)) if recent else 0.0,
                "collision_rate": self.collision_count / max(1, self.total_episodes),
                "success_rate": self.success_count / max(1, self.total_episodes),
            }
            self.metrics_history.append(metrics)
            logger.info(f"[RL] Step {self.num_timesteps}: reward={metrics[\'mean_reward_100\']:.2f}, "
                        f"collision={metrics[\'collision_rate\']:.3f}, success={metrics[\'success_rate\']:.3f}")
        return True

    def get_latest_metrics(self):
        return self.metrics_history[-1] if self.metrics_history else {"timestep":0,"episodes":0,"mean_reward_100":0,"collision_rate":0,"success_rate":0}


class IcebergAvoidanceAgent:
    def __init__(self, hyperparams=None, model_key="default", model_base_dir="/content/arctic/models"):
        self.hyperparams = {**DEFAULT_HYPERPARAMS, **(hyperparams or {})}
        self.model_key = model_key
        self.model_dir = Path(model_base_dir) / f"sac_{model_key}"
        self.model_dir.mkdir(parents=True, exist_ok=True)
        self.model = None
        self.env = None
        self.callback = TrainingMetricsCallback()
        self._model_version = 0

    def create_env(self, difficulty="medium", reward_weights=None, fixed_route=None, fixed_ice_class=None, ship_params=None):
        raw_env = IcebergAvoidanceEnv(difficulty=difficulty, reward_weights=reward_weights,
                                      fixed_route=fixed_route, fixed_ice_class=fixed_ice_class, ship_params=ship_params)
        self.env = RecordEpisodeStatistics(raw_env)
        return self.env

    def build_model(self, difficulty="medium", reward_weights=None):
        if self.env is None: self.create_env(difficulty, reward_weights=reward_weights)
        hp = self.hyperparams
        self.model = SAC("MlpPolicy", self.env,
            learning_rate=hp["learning_rate"], buffer_size=hp["buffer_size"],
            batch_size=hp["batch_size"], gamma=hp["gamma"], tau=hp["tau"],
            ent_coef=hp["ent_coef"], train_freq=hp["train_freq"],
            gradient_steps=hp["gradient_steps"], learning_starts=hp["learning_starts"],
            policy_kwargs=hp["policy_kwargs"], verbose=1, device="auto")
        logger.info("[RL] SAC 모델 생성 완료")
        return self.model

    def train(self, total_timesteps=500_000, extra_callback=None):
        if self.model is None: self.build_model()
        self.callback = TrainingMetricsCallback(log_interval=5000)
        from stable_baselines3.common.callbacks import CallbackList, CheckpointCallback
        checkpoint_cb = CheckpointCallback(save_freq=10_000,
            save_path=str(self.model_dir/"checkpoints"), name_prefix="sac_ckpt", verbose=0)
        cb_list = [self.callback, checkpoint_cb]
        if extra_callback: cb_list.append(extra_callback)
        callbacks = CallbackList(cb_list)
        try:
            self.model.learn(total_timesteps=total_timesteps, callback=callbacks, progress_bar=True)
        except _StopTraining:
            logger.info("[RL] 중단 요청으로 학습 종료")
        finally:
            self._model_version += 1
            self.save()
            logger.info(f"[RL] 모델 저장 완료 (v{self._model_version})")
        return self.callback.get_latest_metrics()

    def predict(self, obs, deterministic=True):
        action, _ = self.model.predict(obs, deterministic=deterministic)
        return action, 0.0

    def save(self, path=None):
        if self.model is None: return
        save_path = path or str(self.model_dir/f"sac_v{self._model_version}")
        self.model.save(save_path)
        logger.info(f"[RL] 모델 저장: {save_path}")

    def load(self, path=None):
        if path:
            load_path = path
        else:
            versions = sorted(self.model_dir.glob("sac_v*.zip"))
            if not versions: return False
            load_path = str(versions[-1]).replace(".zip","")
        try:
            if self.env is None: self.create_env()
            self.model = SAC.load(load_path, env=self.env, device="auto")
            return True
        except Exception as e:
            logger.error(f"[RL] 모델 로드 실패: {e}")
            return False

    def get_training_status(self):
        return {"model_loaded": self.model is not None, "version": self._model_version,
                "metrics": self.callback.get_latest_metrics()}
''')

print('에이전트 모듈 생성 완료')

In [ ]:
# ============================================================
# CELL 7: 트레이너 모듈 작성
# ============================================================
with open('/content/arctic/modules/rl_trainer.py', 'w') as f:
    f.write('''
from __future__ import annotations
import logging, time
from dataclasses import dataclass
from typing import Optional
from stable_baselines3.common.callbacks import BaseCallback
from .rl_agent import IcebergAvoidanceAgent, _StopTraining
from .rl_environment import IcebergAvoidanceEnv
from .rl_reward import RewardWeights

logger = logging.getLogger(__name__)

class _StopCallback(BaseCallback):
    def __init__(self, trainer):
        super().__init__(verbose=0)
        self.trainer = trainer
    def _on_step(self):
        if self.trainer.stop_requested:
            raise _StopTraining("중단 요청")
        return True

@dataclass
class CurriculumStage:
    name: str
    difficulty: str
    timesteps: int
    description: str

CURRICULUM = [
    CurriculumStage("stage_1_basic",    "easy",   50, "빙산 없음, 경로 완주"),
    CurriculumStage("stage_2_moderate", "medium", 33, "빙산 도입, 회피 학습"),
    CurriculumStage("stage_3_hard",     "hard",   17, "고난이도"),
]

class RLTrainer:
    def __init__(self, hyperparams=None, model_key="default", fixed_route=None,
                 fixed_ice_class=None, ship_params=None, model_base_dir="/content/arctic/models"):
        self.agent = IcebergAvoidanceAgent(hyperparams, model_key=model_key, model_base_dir=model_base_dir)
        self._fixed_route = fixed_route
        self._fixed_ice_class = fixed_ice_class
        self._ship_params = ship_params
        self.is_training = False
        self.stop_requested = False
        self.current_stage = None
        self.training_log = []

    def _create_env(self, difficulty, reward_weights=None):
        return self.agent.create_env(difficulty=difficulty, reward_weights=reward_weights,
            fixed_route=self._fixed_route, fixed_ice_class=self._fixed_ice_class, ship_params=self._ship_params)

    def train_curriculum(self, stages=None, reward_weights=None, base_timesteps=None):
        stages = stages or CURRICULUM
        self.is_training = True
        self.stop_requested = False
        results = []
        total_ratio = sum(s.timesteps for s in stages)
        def _stage_ts(stage):
            if base_timesteps is None: return stage.timesteps
            return max(10_000, int(base_timesteps * stage.timesteps / total_ratio))
        try:
            for i, stage in enumerate(stages):
                if self.stop_requested: break
                self.current_stage = stage.name
                ts = _stage_ts(stage)
                logger.info(f"[Trainer] 커리큘럼 {i+1}/{len(stages)}: {stage.name} ({ts:,} steps)")
                self._create_env(stage.difficulty, reward_weights)
                if self.agent.model is None:
                    self.agent.build_model(difficulty=stage.difficulty, reward_weights=reward_weights)
                else:
                    self.agent.model.set_env(self.agent.env)
                start = time.time()
                metrics = self.agent.train(total_timesteps=ts, extra_callback=_StopCallback(self))
                result = {"stage":stage.name,"difficulty":stage.difficulty,
                          "timesteps":ts,"elapsed_seconds":time.time()-start,"metrics":metrics}
                results.append(result)
                self.training_log.append(result)
                if self.stop_requested: break
        finally:
            self.is_training = False
            self.current_stage = None
        return {"stages": results}

    def evaluate(self, n_episodes=100, difficulty="medium"):
        if self.agent.model is None:
            if not self.agent.load(): return {"error": "모델 없음"}
        import numpy as np
        env = IcebergAvoidanceEnv(difficulty=difficulty,
            fixed_route=self._fixed_route, fixed_ice_class=self._fixed_ice_class, ship_params=self._ship_params)
        rewards, collisions, successes = [], 0, 0
        for _ in range(n_episodes):
            if self.stop_requested: break
            obs, _ = env.reset()
            total_reward = 0
            while True:
                action, _ = self.agent.predict(obs, deterministic=True)
                obs, reward, terminated, truncated, info = env.step(action)
                total_reward += reward
                if terminated or truncated:
                    if info.get("collision"): collisions += 1
                    if info.get("success"): successes += 1
                    break
            rewards.append(total_reward)
        return {
            "episodes": len(rewards), "difficulty": difficulty,
            "mean_reward": float(np.mean(rewards)) if rewards else 0.0,
            "collision_rate": collisions/n_episodes if n_episodes>0 else 0.0,
            "success_rate": successes/n_episodes if n_episodes>0 else 0.0,
        }
''')

print('트레이너 모듈 생성 완료')

In [ ]:
# ============================================================
# CELL 8: 모듈 임포트 검증
# ============================================================
import sys
if '/content/arctic' not in sys.path:
    sys.path.insert(0, '/content/arctic')

from modules.rl_environment import IcebergAvoidanceEnv
from modules.rl_trainer import RLTrainer
from modules.rl_reward import RewardWeights

# 환경 동작 확인
env = IcebergAvoidanceEnv(difficulty='easy')
obs, _ = env.reset()
action = env.action_space.sample()
obs2, reward, term, trunc, info = env.step(action)
print(f'환경 OK: obs_shape={obs.shape}, reward={reward:.2f}, collision={info["collision"]}')
print('모든 모듈 임포트 성공')

In [ ]:
# ============================================================
# CELL 9: 학습 설정
# ============================================================
import os

# ---- 학습 조합 ----
ROUTES      = ['NSR', 'NWP', 'TSR']
ICE_CLASSES = ['PC7', 'PC6', 'PC5', 'PC4', 'PC3', 'IA Super', 'IA']
SHIP_TYPES  = {
    'bulk':      {'max_speed_knots': 12.0, 'ice_drag_factor': 0.50},
    'tanker':    {'max_speed_knots': 14.0, 'ice_drag_factor': 0.45},
    'container': {'max_speed_knots': 18.0, 'ice_drag_factor': 0.35},
    'lng':       {'max_speed_knots': 16.0, 'ice_drag_factor': 0.40},
}

# ---- 타임스텝 설정 ----
# T4 GPU 기준: ~5000 fps → 300K steps ≈ 60초/조합 → 84조합 ≈ 84분/iteration
BASE_TIMESTEPS = 300_000   # 조합당 총 타임스텝
N_ITERATIONS   = 3         # 반복 학습 횟수
EVAL_EPISODES  = 30        # 평가 에피소드 수

# ---- 저장 경로 ----
MODEL_BASE_DIR = '/content/arctic/models'
DRIVE_MODEL_DIR = f'{DRIVE_DIR}/models'
os.makedirs(MODEL_BASE_DIR, exist_ok=True)

ALL_COMBOS = [(r, ic, st) for r in ROUTES for ic in ICE_CLASSES for st in SHIP_TYPES]
print(f'학습 조합: {len(ALL_COMBOS)}개')
print(f'조합당 {BASE_TIMESTEPS:,} steps × {N_ITERATIONS} iterations')
print(f'T4 GPU 예상 시간: ~{len(ALL_COMBOS)*BASE_TIMESTEPS//5000//3600:.0f}~{len(ALL_COMBOS)*BASE_TIMESTEPS//3000//3600:.0f}시간/iteration')

In [ ]:
# ============================================================
# CELL 10: 학습 실행 (순차 — 메모리 안정)
# ============================================================
import logging, time, json
from datetime import datetime
from pathlib import Path
from modules.rl_trainer import RLTrainer
from modules.rl_ship_dynamics import ShipParams

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    handlers=[
        logging.FileHandler(f'{DRIVE_DIR}/logs/colab_{datetime.now().strftime("%Y%m%d_%H%M")}.log'),
        logging.StreamHandler(),
    ]
)
logger = logging.getLogger('colab_train')

results_path = f'{DRIVE_DIR}/logs/results.json'
if Path(results_path).exists():
    with open(results_path) as f:
        all_results = json.load(f)
    logger.info(f'이전 결과 로드: {len(all_results)}개 완료')
else:
    all_results = {}

total = len(ALL_COMBOS)
done = 0
start_all = time.time()

for route, ice_class, ship_type in ALL_COMBOS:
    combo_key = f'{route}_{ice_class}_{ship_type}'.replace(' ', '_')

    # 수렴 완료된 조합 스킵
    if combo_key in all_results and all_results[combo_key].get('converged'):
        done += 1
        logger.info(f'[{done}/{total}] SKIP (수렴): {combo_key}')
        continue

    ship_cfg = SHIP_TYPES[ship_type]
    ship_params = ShipParams(max_speed_knots=ship_cfg['max_speed_knots'],
                             ice_drag_factor=ship_cfg['ice_drag_factor'])

    trainer = RLTrainer(
        model_key=combo_key,
        fixed_route=route,
        fixed_ice_class=ice_class,
        ship_params=ship_params,
        model_base_dir=MODEL_BASE_DIR,
    )

    logger.info(f'[{done+1}/{total}] 학습 시작: {combo_key}')
    t0 = time.time()

    best_metrics = {}
    converged = False

    for iteration in range(1, N_ITERATIONS + 1):
        logger.info(f'  Iteration {iteration}/{N_ITERATIONS}')
        trainer.train_curriculum(base_timesteps=BASE_TIMESTEPS)
        metrics = trainer.evaluate(n_episodes=EVAL_EPISODES, difficulty='medium')
        logger.info(f'  평가: success={metrics.get("success_rate",0):.3f}, collision={metrics.get("collision_rate",0):.3f}')
        best_metrics = metrics
        if metrics.get('success_rate', 0) >= 0.70 and metrics.get('collision_rate', 1) <= 0.15:
            converged = True
            logger.info(f'  수렴 달성!')
            break

    elapsed = time.time() - t0
    all_results[combo_key] = {
        'route': route, 'ice_class': ice_class, 'ship_type': ship_type,
        'converged': converged, 'metrics': best_metrics, 'elapsed_sec': elapsed
    }

    # Drive에 중간 저장
    with open(results_path, 'w') as f:
        json.dump(all_results, f, indent=2, ensure_ascii=False)

    # 모델을 Drive에 복사
    import shutil
    src = f'{MODEL_BASE_DIR}/sac_{combo_key}'
    dst = f'{DRIVE_MODEL_DIR}/sac_{combo_key}'
    if Path(src).exists():
        if Path(dst).exists(): shutil.rmtree(dst)
        shutil.copytree(src, dst)

    done += 1
    elapsed_all = (time.time() - start_all) / 3600
    eta = elapsed_all / done * (total - done) if done > 0 else 0
    logger.info(f'[{done}/{total}] 완료: {combo_key} | 수렴={converged} | 경과={elapsed_all:.1f}h | ETA={eta:.1f}h')

converged_count = sum(1 for v in all_results.values() if v.get('converged'))
logger.info(f'=== 전체 완료: {done}/{total}개, 수렴={converged_count}개 ===')

In [ ]:
# ============================================================
# CELL 11: 결과 요약
# ============================================================
import json
from pathlib import Path

results_path = f'{DRIVE_DIR}/logs/results.json'
if Path(results_path).exists():
    with open(results_path) as f:
        all_results = json.load(f)

total = len(all_results)
converged = sum(1 for v in all_results.values() if v.get('converged'))
print(f'완료: {total}개 / 수렴: {converged}개')
print()

print(f'{"조합":<35} {"수렴":>6} {"성공률":>8} {"충돌률":>8}')
print('-'*60)
for key, v in sorted(all_results.items()):
    m = v.get('metrics', {})
    print(f'{key:<35} {str(v.get("converged",False)):>6} '
          f'{m.get("success_rate",0):>8.3f} {m.get("collision_rate",0):>8.3f}')

## 사용 가이드

### 실행 순서
1. **런타임 유형 변경**: 런타임 > 런타임 유형 변경 > **T4 GPU** 선택
2. **셀 1~10 순서대로 실행** (셀 11은 결과 확인용)
3. 결과는 자동으로 Google Drive > `arctic_rl/` 에 저장됨

### 이어하기 (세션 끊긴 후 재개)
- 셀 1~8 실행 (설치 + 모듈 재생성)
- **셀 9부터** 실행 — `results.json` 이 있으면 완료된 조합 자동 스킵

### 예상 시간 (T4 GPU)
| | 시간 |
|--|--|
| 조합당 300K steps | ~1분 |
| 84조합 × 1 iteration | ~1.5시간 |
| 84조합 × 3 iterations | ~4~5시간 |

### 로컬로 결과 가져오기
Google Drive > `arctic_rl/models/` 폴더를 통째로 다운로드 후:
```powershell
# 다운로드한 models 폴더를 로컬 경로에 배치
xcopy /E /I models c:\cccc\Digital_twin\rl-pipeline\models
```